In [1]:
# Import thư viện
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

DATA_RAW = Path("../data/raw")
train_path = DATA_RAW / "application_train.csv"
df_train = pd.read_csv(train_path)
print(f"Kích thước tập train: {df_train.shape}")


from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
import time
print("Successfully Imported")

Kích thước tập train: (307511, 122)
Successfully Imported


In [2]:
"""
Thực hiện lấy ra bộ dữ liệu chứa các features của 5 nhóm quan trọng trước để experiments.

"""
# 1. Biến định danh và biến mục tiêu
ID_COL = 'SK_ID_CURR'
TARGET_COL = 'TARGET'

# 2. Danh sách biến số liên tục (Numerical Features - 11 cột)
NUMERICAL_COLS = [
    # Nhóm 2: Điểm tín dụng ngoài
    'EXT_SOURCE_1',
    'EXT_SOURCE_2',
    'EXT_SOURCE_3',
    
    # Nhóm 3: Năng lực tài chính
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'AMT_GOODS_PRICE',
    
    # Nhóm 4 & 5: Thời gian & Gia đình
    'DAYS_BIRTH',
    'DAYS_EMPLOYED',
    'CNT_CHILDREN',
    'CNT_FAM_MEMBERS'
]

# 3. Danh sách biến danh mục/chữ (Categorical Features - 10 cột)
CATEGORICAL_COLS = [
    # Nhóm 3: Hợp đồng & Thu nhập
    'NAME_CONTRACT_TYPE',
    'NAME_INCOME_TYPE',
    
    # Nhóm 4: Nhân khẩu học
    'CODE_GENDER',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    
    # Nhóm 5: Công việc & Nơi ở
    'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE',
    'NAME_HOUSING_TYPE',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY'
]

# 4. Danh sách tổng hợp tất cả 23 cột (Dùng để lọc DataFrame ban đầu)
ALL_SELECTED_COLS = [ID_COL, TARGET_COL] + NUMERICAL_COLS + CATEGORICAL_COLS

In [3]:
df_sub = df_train[ALL_SELECTED_COLS].copy()
print(df_sub.shape)


(307511, 23)


In [4]:
df_clean = df_sub.copy()

# 2. Xử lý Tuổi (DAYS_BIRTH -> AGE)
df_clean['AGE'] = -df_clean['DAYS_BIRTH'] / 365

# 3. Xử lý Ngày làm việc (DAYS_EMPLOYED)
# Tạo cờ đánh dấu người về hưu/thất nghiệp
df_clean['DAYS_EMPLOYED_ANOM'] = (df_clean['DAYS_EMPLOYED'] == 365243).astype(int)

# Thay 365243 thành NaN và đổi sang số năm dương
df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].replace({365243: np.nan})
df_clean['YEARS_EMPLOYED'] = -df_clean['DAYS_EMPLOYED'] / 365

# 4. Xóa 2 cột cũ để tránh dư thừa dữ liệu
df_clean = df_clean.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'])

# 5. Cập nhật lại danh sách NUMERICAL_COLS (Bỏ cột cũ, thêm cột mới)
NUMERICAL_COLS_UPDATED = [col for col in NUMERICAL_COLS if col not in ['DAYS_BIRTH', 'DAYS_EMPLOYED']]
NUMERICAL_COLS_UPDATED += ['AGE', 'YEARS_EMPLOYED', 'DAYS_EMPLOYED_ANOM']

print("Danh sách biến số mới:", NUMERICAL_COLS_UPDATED)

Danh sách biến số mới: ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AGE', 'YEARS_EMPLOYED', 'DAYS_EMPLOYED_ANOM']


In [5]:
""" 
Thực hiện chia tập dữ liệu ->  train/valid trước 

"""
X = df_clean.drop(columns = [ID_COL, TARGET_COL])
y = df_clean[TARGET_COL]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size= 0.1, random_state= 23102026, stratify = y)

X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size= 0.1, random_state= 23102026, stratify = y_temp)

print(f"Kích thước X_train: {X_train.shape} (Tỷ lệ nợ xấu: {y_train.mean():.4f})")
print(f"Kích thước X_valid: {X_valid.shape} (Tỷ lệ nợ xấu: {y_valid.mean():.4f})")
print(f"Kích thước X_test:  {X_test.shape}  (Tỷ lệ nợ xấu: {y_test.mean():.4f})")

Kích thước X_train: (249083, 22) (Tỷ lệ nợ xấu: 0.0807)
Kích thước X_valid: (27676, 22) (Tỷ lệ nợ xấu: 0.0807)
Kích thước X_test:  (30752, 22)  (Tỷ lệ nợ xấu: 0.0807)


In [6]:
""" 
Thực hiện Feature Enginner
Fit transform trên tập train và transform trên tập valid/ test. 

Các bước thực hiện: 

+ Đối với các biến có giá trị, ta dùng meadian để fit vào các vị trí NA

+ Đối với các biến categorical, ta tạo thêm label "Missing" để gán vào vị trí đó rồi mới CategoricalENcoder.

"""

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median"))
])

cat_pipeline =  Pipeline([
    ('imputer', SimpleImputer(strategy = "constant", fill_value= "Missing")),
    ('onehot', OneHotEncoder(handle_unknown= "ignore", sparse_output= False))
])

# thuc hien qua hai luong xu li

preprocessor = ColumnTransformer(transformers = [
    ('num', num_pipeline, NUMERICAL_COLS_UPDATED), 
    ('cat', cat_pipeline, CATEGORICAL_COLS)
])


X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train sau khi biến đổi: {X_train_processed.shape}")
print(f"X_valid sau khi biến đổi: {X_valid_processed.shape}")
print(f"X_test sau khi biến đổi:  {X_test_processed.shape}")


X_train sau khi biến đổi: (249083, 123)
X_valid sau khi biến đổi: (27676, 123)
X_test sau khi biến đổi:  (30752, 123)


In [7]:
# debugg
# kiểm tra đã preprocessing chính xác chưa 

if __name__ == "__main__" : 
    import numpy as np
    import pandas as pd

    print("=== 1. KIỂM TRA GIÁ TRỊ KHUYẾT (NAN CHECK) ===")
    nan_train = np.isnan(X_train_processed).sum()
    nan_valid = np.isnan(X_valid_processed).sum()
    nan_test  = np.isnan(X_test_processed).sum()
    print(f"Số lượng NaN còn sót lại trong Train: {nan_train}")
    print(f"Số lượng NaN còn sót lại trong Valid: {nan_valid}")
    print(f"Số lượng NaN còn sót lại trong Test:  {nan_test}")
    assert nan_train == 0 and nan_valid == 0 and nan_test == 0, "Vẫn còn giá trị NaN!"
    print("-> ĐẠT: Toàn bộ dữ liệu đã sạch 100% NaN!\n")


    print("=== 2. KIỂM TRA DANH SÁCH 123 CỘT ĐƯỢC TẠO RA ===")
    # Lấy tên cột thực tế mà preprocessor đã tạo ra
    feature_names = preprocessor.get_feature_names_out()
    print(f"Tổng số đặc trưng: {len(feature_names)}")
    print("10 cột đầu tiên:", feature_names[:10])
    print("10 cột cuối cùng:", feature_names[-10:])
    print()


    print("=== 3. KIỂM TRA LOGIC TUỔI VÀ NGÀY LÀM VIỆC ===")
    # Tìm vị trí cột AGE và YEARS_EMPLOYED
    df_check = pd.DataFrame(X_train_processed, columns=feature_names)
    print(f"Tuổi nhỏ nhất: {df_check['num__AGE'].min():.1f} | Lớn nhất: {df_check['num__AGE'].max():.1f}")
    print(f"Năm làm việc nhỏ nhất: {df_check['num__YEARS_EMPLOYED'].min():.1f} | Lớn nhất: {df_check['num__YEARS_EMPLOYED'].max():.1f}")
    # Kiểm tra xem còn ai có số năm làm việc 1.000 năm không:
    assert df_check['num__YEARS_EMPLOYED'].max() < 70, "Vẫn còn người làm việc > 70 năm!"
    print("-> ĐẠT: Logic số tuổi và năm làm việc hoàn toàn hợp lý (không còn số âm, không còn 1000 năm)!\n")


    print("=== 4. KIỂM TRA CỘT MÃ HÓA ONE-HOT (CHỈ ĐƯỢC CHỨA 0 VÀ 1) ===")
    # Lấy các cột One-hot (bắt đầu bằng 'cat__')
    cat_cols = [c for c in feature_names if c.startswith('cat__')]
    unique_vals = np.unique(df_check[cat_cols].values)
    print(f"Các giá trị xuất hiện trong các cột One-Hot: {unique_vals}")
    assert set(unique_vals).issubset({0.0, 1.0}), "Cột One-Hot chứa giá trị khác 0 và 1!"
    print("-> ĐẠT: Toàn bộ biến danh mục đã chuyển thành nhị phân 0 và 1 chuẩn xác!")

=== 1. KIỂM TRA GIÁ TRỊ KHUYẾT (NAN CHECK) ===
Số lượng NaN còn sót lại trong Train: 0
Số lượng NaN còn sót lại trong Valid: 0
Số lượng NaN còn sót lại trong Test:  0
-> ĐẠT: Toàn bộ dữ liệu đã sạch 100% NaN!

=== 2. KIỂM TRA DANH SÁCH 123 CỘT ĐƯỢC TẠO RA ===
Tổng số đặc trưng: 123
10 cột đầu tiên: ['num__EXT_SOURCE_1' 'num__EXT_SOURCE_2' 'num__EXT_SOURCE_3'
 'num__AMT_INCOME_TOTAL' 'num__AMT_CREDIT' 'num__AMT_ANNUITY'
 'num__AMT_GOODS_PRICE' 'num__CNT_CHILDREN' 'num__CNT_FAM_MEMBERS'
 'num__AGE']
10 cột cuối cùng: ['cat__NAME_HOUSING_TYPE_Co-op apartment'
 'cat__NAME_HOUSING_TYPE_House / apartment'
 'cat__NAME_HOUSING_TYPE_Municipal apartment'
 'cat__NAME_HOUSING_TYPE_Office apartment'
 'cat__NAME_HOUSING_TYPE_Rented apartment'
 'cat__NAME_HOUSING_TYPE_With parents' 'cat__FLAG_OWN_CAR_N'
 'cat__FLAG_OWN_CAR_Y' 'cat__FLAG_OWN_REALTY_N' 'cat__FLAG_OWN_REALTY_Y']

=== 3. KIỂM TRA LOGIC TUỔI VÀ NGÀY LÀM VIỆC ===
Tuổi nhỏ nhất: 20.5 | Lớn nhất: 69.0
Năm làm việc nhỏ nhất: -0.0 | Lớn nhất: 

In [8]:
start_time = time.time()

baseline_model = LGBMClassifier(
    n_estimators = 100, 
    learning_rate = 0.04,
    random_state = 23102026,
    n_jobs = -1,
    verbose = 1
)

baseline_model.fit(X_train_processed, y_train)

train_time = time.time() - start_time

# Prediction in Train/Valid/ Test Set

y_train_pred = baseline_model.predict_proba(X_train_processed)[:, 1]
y_valid_pred = baseline_model.predict_proba(X_valid_processed)[:, 1]
y_test_pred = baseline_model.predict_proba(X_test_processed)[:, 1]
# Tính toán ROC-AUC
auc_train = roc_auc_score(y_train, y_train_pred)
auc_valid = roc_auc_score(y_valid, y_valid_pred)
auc_test  = roc_auc_score(y_test, y_test_pred)
# Tính Hệ số Gini chuẩn ngân hàng (Gini = 2 * AUC - 1)
gini_train = 2 * auc_train - 1
gini_valid = 2 * auc_valid - 1
gini_test  = 2 * auc_test - 1

print("     KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL        ")
print("=" * 45)
print(f"Tập Huấn luyện (Train): AUC = {auc_train:.4f} | Gini = {gini_train:.4f}")
print(f"Tập Kiểm định  (Valid): AUC = {auc_valid:.4f} | Gini = {gini_valid:.4f}")
print(f"Tập Kiểm tra    (Test): AUC = {auc_test:.4f} | Gini = {gini_test:.4f}")

[LightGBM] [Info] Number of positive: 20108, number of negative: 228975
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010249 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2471
[LightGBM] [Info] Number of data points in the train set: 249083, number of used features: 117
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080728 -> initscore=-2.432495
[LightGBM] [Info] Start training from score -2.432495
     KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL        
Tập Huấn luyện (Train): AUC = 0.7651 | Gini = 0.5303
Tập Kiểm định  (Valid): AUC = 0.7479 | Gini = 0.4958
Tập Kiểm tra    (Test): AUC = 0.7461 | Gini = 0.4922


In [9]:
""""
Feature Engineering

+ Tạo thêm cột EXT_SOURCE_MEAN : Lấy trung bình 3 cột EXT_SOURCE lại, 
+ PAYMENT_RATE  = AMT_ANNUITY / AMT_CREDIT : Tỉ lệ trả nợ trên tổng vay
+ INCOME_CREDIT_PERC = AMT_INCOME_TOTAL / AMT_CREDIT: Thu nhập gánh được bao nhiêu % khoản vay.
+ ANNUITY_INCOME_PERC = AMT_ANNUITY / AMT_INCOME_TOTAL: Tiền góp mỗi kỳ chiếm bao nhiêu % thu nhập tháng.

"""
df_clean["EXT_SOURCES_MEAN"] = df_clean[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df_clean["PAYMENT_RATE"] = df_clean["AMT_ANNUITY"]/ df_clean["AMT_CREDIT"]
df_clean["INCOME_CREDIT_PERC"] = df_clean["AMT_INCOME_TOTAL"] /df_clean["AMT_CREDIT"]
df_clean["ANNUITY_INCOME_PERC"] = df_clean["AMT_ANNUITY"] / df_clean["AMT_INCOME_TOTAL"]
NUMERICAL_COLS_UPDATED += ["EXT_SOURCES_MEAN", "PAYMENT_RATE", "INCOME_CREDIT_PERC", "ANNUITY_INCOME_PERC"]

# chia tập dữ liệu 
X = df_clean.drop(columns = [ID_COL, TARGET_COL])
y = df_clean[TARGET_COL]
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size= 0.1, random_state= 23102026, stratify = y)
X_train, X_valid, y_train, y_valid = train_test_split(X_temp, y_temp, test_size= 0.1, random_state= 23102026, stratify = y_temp)

# preprocessing 
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median"))
])

cat_pipeline =  Pipeline([
    ('imputer', SimpleImputer(strategy = "constant", fill_value= "Missing")),
    ('onehot', OneHotEncoder(handle_unknown= "ignore", sparse_output= False))
])

# thuc hien qua hai luong xu li

preprocessor = ColumnTransformer(transformers = [
    ('num', num_pipeline, NUMERICAL_COLS_UPDATED), 
    ('cat', cat_pipeline, CATEGORICAL_COLS)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print(f"Số lượng đặc trưng sau khi thêm Feature mới: {X_train_processed.shape[1]}")

start_time = time.time()

baseline_model = LGBMClassifier(
    n_estimators = 100, 
    learning_rate = 0.04,
    random_state = 23102026,
    n_jobs = -1,
    verbose = -1
)

baseline_model.fit(X_train_processed, y_train)

train_time = time.time() - start_time

# Prediction in Train/Valid/ Test Set

y_train_pred = baseline_model.predict_proba(X_train_processed)[:, 1]
y_valid_pred = baseline_model.predict_proba(X_valid_processed)[:, 1]
y_test_pred = baseline_model.predict_proba(X_test_processed)[:, 1]
# Tính toán ROC-AUC
auc_train = roc_auc_score(y_train, y_train_pred)
auc_valid = roc_auc_score(y_valid, y_valid_pred)
auc_test  = roc_auc_score(y_test, y_test_pred)
# Tính Hệ số Gini chuẩn ngân hàng (Gini = 2 * AUC - 1)
gini_train = 2 * auc_train - 1
gini_valid = 2 * auc_valid - 1
gini_test  = 2 * auc_test - 1

print("     KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL        ")
print("=" * 45)
print(f"Tập Huấn luyện (Train): AUC = {auc_train:.4f} | Gini = {gini_train:.4f}")
print(f"Tập Kiểm định  (Valid): AUC = {auc_valid:.4f} | Gini = {gini_valid:.4f}")
print(f"Tập Kiểm tra    (Test): AUC = {auc_test:.4f} | Gini = {gini_test:.4f}")


Số lượng đặc trưng sau khi thêm Feature mới: 127
     KẾT QUẢ ĐÁNH GIÁ BASELINE MODEL        
Tập Huấn luyện (Train): AUC = 0.7719 | Gini = 0.5438
Tập Kiểm định  (Valid): AUC = 0.7573 | Gini = 0.5146
Tập Kiểm tra    (Test): AUC = 0.7509 | Gini = 0.5018


## LÍ DO KHÔNG LỰA CHỌN CÁC FEATURES CÒN LẠI

+ Missing quá cao (>60% ở 40 cột căn hộ).
+ Phương sai bằng 0 (20 cột giấy tờ hầu hết là 0).
+ Đa cộng tuyến (lặp lại _AVG, _MODE, _MEDI).
+ Hiệu quả kinh doanh (ROI): Giữ mô hình tinh gọn, chạy nhanh (<10ms trên API), dễ giải thích (Explainable AI), và không làm phiền khách hàng khi điền hồ sơ.

In [ ]:
from pathlib import Path

# 1. Đường dẫn thư mục lưu trữ
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Định nghĩa danh sách đúng 27 cột khớp 100% với src/config.py
CSV_COLUMNS = [
    # 16 biến số
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AGE', 'YEARS_EMPLOYED',
    'DAYS_EMPLOYED_ANOM', 'EXT_SOURCES_MEAN', 'PAYMENT_RATE',
    'INCOME_CREDIT_PERC', 'ANNUITY_INCOME_PERC',
    # 10 biến danh mục
    'NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 'CODE_GENDER',
    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'OCCUPATION_TYPE',
    'ORGANIZATION_TYPE', 'NAME_HOUSING_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    # 1 biến target
    'TARGET'
]

# 3. Ghép X và y lại rồi sắp xếp đúng thứ tự cột
train_df = X_train.copy()
train_df['TARGET'] = y_train
train_df = train_df[CSV_COLUMNS]

valid_df = X_valid.copy()
valid_df['TARGET'] = y_valid
valid_df = valid_df[CSV_COLUMNS]

test_df = X_test.copy()
test_df['TARGET'] = y_test
test_df = test_df[CSV_COLUMNS]

# 4. Xuất ra 3 file CSV trong data/processed/
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
valid_df.to_csv(PROCESSED_DIR / "valid.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Đã xuất thành công 3 file vào data/processed/!")
print(f"Train shape: {train_df.shape}")
print(f"Valid shape: {valid_df.shape}")
print(f"Test shape:  {test_df.shape}")

✅ Đã xuất thành công 3 file vào data/processed/!
Train shape: (249083, 27)
Valid shape: (27676, 27)
Test shape:  (30752, 27)
